In [7]:
import pandas as pd
import re
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

print("⏳ Warming up the engine: Loading match data and crafting our 7 Golden Features...")

# 1. Load the consolidated dataset gathered across all major international tournaments
df = pd.read_csv('data.csv')

rows = []

# 2. Deconstruct each match into two distinct perspectives (Home team & Away team)
# This dual-perspective approach doubles our training data and captures both tactical angles
for idx, row in df.iterrows():
    # Clean up score strings to handle edge cases (like penalty shootouts or dashes)
    clean_score = re.sub(r'\(.*?\)', '', str(row['score'])).replace('–', '-').strip()
    try:
        parts = clean_score.split('-')
        h_score, a_score = int(parts[0].strip()), int(parts[1].strip())
    except:
        # Skip gracefully if the score format is missing or corrupted
        continue

    # Calculate passing efficiency safely to avoid zero-division issues
    h_pass_acc = (row['home_completed_passes'] / row['home_attempted_pases']) * 100 if row['home_attempted_pases'] > 0 else 0
    a_pass_acc = (row['away_completed_passes'] / row['away_attempted_pases']) * 100 if row['away_attempted_pases'] > 0 else 0

    # PPDA (Passes Allowed Per Defensive Action) - captures pressing aggression
    # We use max(..., 1) as a safety net against dividing by zero
    h_ppda = row['away_completed_passes'] / max(row['home_tackles'] + row['home_interceptions'], 1)
    a_ppda = row['home_completed_passes'] / max(row['away_tackles'] + row['away_interceptions'], 1)

    # Ground truth outcomes for the match
    h_outcome = 'Win' if h_score > a_score else ('Loss' if h_score < a_score else 'Draw')
    a_outcome = 'Win' if a_score > h_score else ('Loss' if a_score < h_score else 'Draw')

    # Append home team feature vector
    rows.append({
        'xg': float(row['home_xg']), 'possession': float(row['home_possession']),
        'shots_on_target': float(row['home_sot']), 'pass_accuracy': h_pass_acc,
        'ppda': h_ppda, 'tackles_successful': float(row['home_tackles']),
        'interceptions': float(row['home_interceptions']), 'outcome': h_outcome
    })

    # Append away team feature vector
    rows.append({
        'xg': float(row['away_xg']), 'possession': float(row['away_possession']),
        'shots_on_target': float(row['away_sot']), 'pass_accuracy': a_pass_acc,
        'ppda': a_ppda, 'tackles_successful': float(row['away_tackles']),
        'interceptions': float(row['away_interceptions']), 'outcome': a_outcome
    })

clean_df = pd.DataFrame(rows)
print(f"✅ Data processing complete! Extracted {len(clean_df)} tactical feature samples.")

# Separate input features (X) and target labels (y)
X = clean_df[['xg', 'possession', 'shots_on_target', 'pass_accuracy', 'ppda', 'tackles_successful', 'interceptions']]
y = clean_df['outcome']

# XGBoost strictly requires numeric target labels (0, 1, 2), so we encode our categorical targets
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Define the three benchmark algorithms matching our project research
algorithms = {
    # Random Forest: Robust baseline with bagging ensemble to handle tabular noise
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),

    # Multi-Layer Perceptron (ANN): Scaled pipeline to uncover deep non-linear patterns
    "Multi-Layer Perceptron": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), max_iter=1000, random_state=42)),

    # XGBoost: State-of-the-art gradient boosted decision tree classifier
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='mlogloss')
}

# 4. Stratified 5-Fold Cross-Validation ensures balanced outcome distributions across all splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n🏟️ Welcome to the Algorithm Arena! Commencing 5-Fold Cross Validation...\n" + "-"*75)

for name, model in algorithms.items():
    # Evaluate generalization performance on the encoded labels
    scores = cross_val_score(model, X, y_encoded, cv=cv, scoring='accuracy')
    print(f"{name:<25} | Average Accuracy: {scores.mean() * 100:.2f}% (Stability: ±{scores.std() * 100:.2f}%)")

print("-" * 75)

⏳ Warming up the engine: Loading match data and crafting our 7 Golden Features...
✅ Data processing complete! Extracted 524 tactical feature samples.

🏟️ Welcome to the Algorithm Arena! Commencing 5-Fold Cross Validation...
---------------------------------------------------------------------------
Random Forest             | Average Accuracy: 58.02% (Stability: ±4.51%)
Multi-Layer Perceptron    | Average Accuracy: 57.08% (Stability: ±4.14%)
XGBoost                   | Average Accuracy: 53.26% (Stability: ±5.53%)
---------------------------------------------------------------------------


In [6]:
import pandas as pd
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("⏳ Initiating the Ultimate Time-Machine Test (Time-Based Split)...")

# 1. Load the unified dataset
df = pd.read_csv('data.csv')
rows = []

# 2. Extract features and keep track of the tournament timeline
for idx, row in df.iterrows():
    clean_score = re.sub(r'\(.*?\)', '', str(row['score'])).replace('–', '-').strip()
    try:
        parts = clean_score.split('-')
        h_score, a_score = int(parts[0].strip()), int(parts[1].strip())
    except:
        continue

    h_pass_acc = (row['home_completed_passes'] / row['home_attempted_pases']) * 100 if row['home_attempted_pases'] > 0 else 0
    a_pass_acc = (row['away_completed_passes'] / row['away_attempted_pases']) * 100 if row['away_attempted_pases'] > 0 else 0

    h_ppda = row['away_completed_passes'] / max(row['home_tackles'] + row['home_interceptions'], 1)
    a_ppda = row['home_completed_passes'] / max(row['away_tackles'] + row['away_interceptions'], 1)

    total_aerials = max(row['home_aerials_won'] + row['away_aerials_won'], 1)
    h_aerial = (row['home_aerials_won'] / total_aerials) * 100
    a_aerial = (row['away_aerials_won'] / total_aerials) * 100

    h_outcome = 'Win' if h_score > a_score else ('Loss' if h_score < a_score else 'Draw')
    a_outcome = 'Win' if a_score > h_score else ('Loss' if a_score < h_score else 'Draw')

    # We attach the 'tournament' label to know if this is past or future data
    tournament = row.get('tournament', 'Unknown')

    rows.append({
        'tournament': tournament, 'xg': float(row['home_xg']), 'possession': float(row['home_possession']),
        'shots_on_target': float(row['home_sot']), 'pass_accuracy': h_pass_acc, 'ppda': h_ppda,
        'tackles_successful': float(row['home_tackles']), 'interceptions': float(row['home_interceptions']),
        'aerial_duels_won_pct': h_aerial, 'outcome': h_outcome
    })

    rows.append({
        'tournament': tournament, 'xg': float(row['away_xg']), 'possession': float(row['away_possession']),
        'shots_on_target': float(row['away_sot']), 'pass_accuracy': a_pass_acc, 'ppda': a_ppda,
        'tackles_successful': float(row['away_tackles']), 'interceptions': float(row['away_interceptions']),
        'aerial_duels_won_pct': a_aerial, 'outcome': a_outcome
    })

clean_df = pd.DataFrame(rows)

# 3. Split the data into PAST (Training) and FUTURE (Testing)
# Past Tournaments: 2018 World Cup & Euro 2020
train_df = clean_df[clean_df['tournament'].isin(['World Cup 2018', 'Euro 2020'])]

# Future Tournaments: 2022 World Cup, Euro 2024, Copa America 2024
test_df = clean_df[clean_df['tournament'].isin(['World Cup 2022', 'Euro 2024', 'Copa America 2024'])]

features = ['xg', 'possession', 'shots_on_target', 'pass_accuracy', 'ppda', 'tackles_successful', 'interceptions', 'aerial_duels_won_pct']

X_train, y_train = train_df[features], train_df['outcome']
X_test, y_test = test_df[features], test_df['outcome']

print(f"📚 Training on {len(X_train)} historical samples (2018-2020)...")
print(f"🔮 Predicting {len(X_test)} future samples (2022-2024)...")

# 4. Train the Random Forest Champion
rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_model.fit(X_train, y_train)

# 5. Evaluate on the future dataset
train_acc = rf_model.score(X_train, y_train)
test_pred = rf_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)

print("\n" + "="*50)
print(f"📈 Training Accuracy (Past):  {train_acc * 100:.2f}%")
print(f"🎯 Testing Accuracy (Future): {test_acc * 100:.2f}%")
print("="*50)

⏳ Initiating the Ultimate Time-Machine Test (Time-Based Split)...
📚 Training on 230 historical samples (2018-2020)...
🔮 Predicting 294 future samples (2022-2024)...

📈 Training Accuracy (Past):  73.04%
🎯 Testing Accuracy (Future): 58.16%


In [3]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.ensemble import RandomForestClassifier

print("⏳ Starting the factory: Building the final AI brain for our app...")

# 1. Loading the massive 500+ match dataset we just aggregated
df = pd.read_csv('data.csv')

rows = []

# 2. Extracting the 7 Golden Features for both Home and Away perspectives
for idx, row in df.iterrows():
    # Cleaning up the scoreline to grab regular time goals
    clean_score = re.sub(r'\(.*?\)', '', str(row['score'])).replace('–', '-').strip()
    try:
        parts = clean_score.split('-')
        h_score, a_score = int(parts[0].strip()), int(parts[1].strip())
    except:
        continue # Skip corrupted scorelines

    # Calculating advanced metrics safely
    h_pass_acc = (row['home_completed_passes'] / row['home_attempted_pases']) * 100 if row['home_attempted_pases'] > 0 else 0
    a_pass_acc = (row['away_completed_passes'] / row['away_attempted_pases']) * 100 if row['away_attempted_pases'] > 0 else 0

    h_ppda = row['away_completed_passes'] / max(row['home_tackles'] + row['home_interceptions'], 1)
    a_ppda = row['home_completed_passes'] / max(row['away_tackles'] + row['away_interceptions'], 1)

    # We must calculate aerial duel win percentage to match the Streamlit UI expectations
    total_aerials = max(row['home_aerials_won'] + row['away_aerials_won'], 1)
    h_aerial = (row['home_aerials_won'] / total_aerials) * 100
    a_aerial = (row['away_aerials_won'] / total_aerials) * 100

    # Determine the match outcome
    h_outcome = 'Win' if h_score > a_score else ('Loss' if h_score < a_score else 'Draw')
    a_outcome = 'Win' if a_score > h_score else ('Loss' if a_score < h_score else 'Draw')

    # Log the home perspective
    rows.append({
        'xg': float(row['home_xg']), 'possession': float(row['home_possession']),
        'shots_on_target': float(row['home_sot']), 'pass_accuracy': h_pass_acc,
        'ppda': h_ppda, 'tackles_successful': float(row['home_tackles']),
        'interceptions': float(row['home_interceptions']), 'aerial_duels_won_pct': h_aerial,
        'outcome': h_outcome
    })

    # Log the away perspective
    rows.append({
        'xg': float(row['away_xg']), 'possession': float(row['away_possession']),
        'shots_on_target': float(row['away_sot']), 'pass_accuracy': a_pass_acc,
        'ppda': a_ppda, 'tackles_successful': float(row['away_tackles']),
        'interceptions': float(row['away_interceptions']), 'aerial_duels_won_pct': a_aerial,
        'outcome': a_outcome
    })

# We save the cleaned data just in case we need to inspect it later
clean_df = pd.DataFrame(rows)
clean_df.to_csv('clean_master_dataset.csv', index=False)
print(f"✅ Data processed! Cleaned master dataset exported with {len(clean_df)} samples.")

# 3. Training the undisputed champion: Random Forest
# We use the hyperparameters that proved to be the most stable in our Arena test
golden_features = [
    'xg', 'possession', 'shots_on_target',
    'ppda', 'tackles_successful', 'interceptions', 'aerial_duels_won_pct'
]

X = clean_df[golden_features]
y = clean_df['outcome']

# Equipping the model with depth limits to prevent overfitting
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X, y)

# 4. Packaging the brain for deployment
joblib.dump(rf_model, 'world_cup_rf_model_v2.pkl')
print("✅ Mission accomplished! The new, highly intelligent world_cup_rf_model_v2.pkl is ready for Streamlit deployment.")

⏳ Starting the factory: Building the final AI brain for our app...
✅ Data processed! Cleaned master dataset exported with 524 samples.
✅ Mission accomplished! The new, highly intelligent world_cup_rf_model_v2.pkl is ready for Streamlit deployment.
